In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, APPNP, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import itertools
import wandb

# Preprocessing Metadata

In [ ]:
metadata_df = pd.read_csv("hiba_metadata.csv")
metadata_df.columns = metadata_df.columns.str.strip().str.lower()

metadata_df.head()

In [ ]:
non_feature_df = metadata_df.drop(['isic_id'], axis=1)
non_feature_df.head()

In [7]:
X = non_feature_df.drop('target', axis=1)
y = non_feature_df['target']

# Identify column types
numeric_cols     = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

# Handle numeric columns
X_num = X[numeric_cols].copy()

# Impute with mean
for col in numeric_cols:
    mean_val = X_num[col].mean()
    X_num[col] = X_num[col].fillna(mean_val)
    
# Scale to zero mean, unit variance
scaler = StandardScaler()

X_num_scaled = pd.DataFrame(
    scaler.fit_transform(X_num),
    columns=numeric_cols,
    index=X.index
)

# Handle categorical columns
X_cat = X[categorical_cols].copy()

# Impute with mode
for col in categorical_cols:
    mode_val = X_cat[col].mode()[0]
    X_cat[col] = X_cat[col].fillna(mode_val)
    
# One-hot encode
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
cat_encoded = encoder.fit_transform(X_cat)
cat_feature_names = encoder.get_feature_names_out(categorical_cols)

X_cat_encoded = pd.DataFrame(cat_encoded, columns=cat_feature_names, index=X.index)

# Concatenate processed features
X_process_df = pd.concat([X_num_scaled, X_cat_encoded], axis=1)

In [ ]:
X_process_df.head()

In [11]:
# Extract label column
y = non_feature_df['target'].copy()

# Find unique labels
unique_labels = y.dropna().unique().tolist()

# Create a mapping dictionary label to integer
label_map = {label: idx for idx, label in enumerate(unique_labels)}

# Map the labels to integers, fill missing labels with -1 (or any number you prefer)
y_encoded = y.map(label_map).fillna(-1).astype(int)

label_dict = {
    idx: label
    for idx, label in y_encoded.items()
}

# Graph Data

In [14]:
NUM_NODES = 8
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 1e-3

In [16]:
def graph_data(image_feat: torch.Tensor, label: int = None):

    if not isinstance(image_feat, torch.Tensor):
        image_feat = torch.tensor(image_feat, dtype=torch.float32)
   
    assert image_feat.shape[0] == 4096

    node_feats = image_feat.view(NUM_NODES, 512)

    num_nodes = node_feats.shape[0]
    
    edges = list(itertools.permutations(range(num_nodes), 2))  # fully connected except self-loops
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    self_loops = torch.arange(num_nodes).unsqueeze(0).repeat(2, 1)
    
    edge_index = torch.cat([edge_index, self_loops], dim=1)

    data = Data(x=node_feats, edge_index=edge_index)
    
    if label is not None:
        data.y = torch.tensor(label, dtype=torch.long)
        
    return data

# Image Features

In [ ]:
with open('hiba_features_labels_filenames.pkl', 'rb') as f:
        data = pickle.load(f)

features = data['features']
filenames = [f.strip().lower() for f in data['filenames']]

img_feats_dict = {fn: feat for fn, feat in zip(filenames, features)}

#metadata_df['isic_id'] = metadata_df['isic_id'].str.strip().str.lower()

matches = sum([row['isic_id'][0] in img_feats_dict for _, row in metadata_df.iterrows()])

print(f"Matching isic_ids after conversion: {matches}")

In [21]:
# Use this just once to clean your metadata_df
metadata_df['isic_id'] = metadata_df['isic_id'].str.strip().str.lower()

encoder = OrdinalEncoder()
y = encoder.fit_transform(metadata_df[['target']])

# Fix: use cleaned 'isic_id' from DataFrame
label_dict = {
    row['isic_id']: int(y[i][0])
    for i, row in metadata_df.iterrows()
}

meta_tensor_dict = {
    row['isic_id']: torch.tensor(X_process_df.iloc[i].values, dtype=torch.float32)
    for i, row in metadata_df.iterrows()
}

In [ ]:
img_ids = set(img_feats_dict.keys())
meta_ids = set(meta_tensor_dict.keys())
label_ids = set(label_dict.keys())

matching_ids = img_ids & meta_ids & label_ids
print(f"Fully matched IDs across all 3 sources: {len(matching_ids)}")

In [ ]:
data_list = []

for isic_id in matching_ids:
    vec = img_feats_dict[isic_id]
    label = label_dict[isic_id]
    meta = meta_tensor_dict[isic_id]

    graph = graph_data(vec, label)
    graph.meta = meta
    data_list.append(graph)

print(f"Total graphs created: {len(data_list)}")

In [27]:
# Train 90%
train_data, temp_data = train_test_split(data_list, test_size=0.2, random_state=42, stratify=[data.y.item() for data in data_list])

# Test 10%
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42, stratify=[data.y.item() for data in temp_data])

# GNN Model

In [30]:
class CustomSAGE_APPNP(nn.Module):
    def __init__(self, node_feat_dim, meta_feat_dim, hidden_channels=256, out_channels=3, K=10, alpha=0.1, dropout_p=0.6):
        super().__init__()

        self.input_proj = nn.Linear(node_feat_dim, hidden_channels)

        self.sage1 = SAGEConv(hidden_channels, hidden_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)

        self.appnp = APPNP(K=K, alpha=alpha)

        self.dropout = nn.Dropout(dropout_p)

        # Project metadata
        self.meta_proj = nn.Linear(meta_feat_dim, hidden_channels)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels + hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_channels, out_channels)
        )

    def forward(self, x, x_meta, edge_index, batch):
        # GNN layers
        x = self.input_proj(x)
        x = self.sage1(x, edge_index)
        x = F.relu(x)
        
        # APPNP Layer
        x = self.appnp(x, edge_index)
        
        # Graph-level embedding
        g = global_mean_pool(x, batch)  # [num_graphs, hidden_channels]
        
        # Handle metadata
        num_graphs = batch.max().item() + 1
        num_nodes_in_batch = batch.size(0)
        
        # Ensure metadata is 2D
        if x_meta.dim() == 1:
            x_meta = x_meta.unsqueeze(-1)  # [N, 1]
        
        # Case 1: Metadata matches batch node count
        if x_meta.size(0) == num_nodes_in_batch:
            x_meta = global_mean_pool(x_meta, batch)  # [num_graphs, 1]
        
        # Case 2: Metadata already matches graph count
        elif x_meta.size(0) == num_graphs:
            pass  # Already correct shape
        
        # Case 3: Metadata is for full dataset (larger than batch)
        elif x_meta.size(0) > num_nodes_in_batch:
            # Select only the nodes present in this batch
            # Assuming batch indices correspond to original dataset ordering
            x_meta = x_meta[:num_nodes_in_batch]  # Take first N nodes
            x_meta = global_mean_pool(x_meta, batch)  # [num_graphs, 1]
        
        else:
            raise ValueError(
                f"Metadata shape {x_meta.shape} incompatible with:\n"
                f"- Batch node count: {num_nodes_in_batch}\n"
                f"- Number of graphs: {num_graphs}"
            )
        
        # Project metadata features
        x_meta = self.meta_proj(x_meta)  # [num_graphs, hidden_channels]
        
        # Concatenate and classify
        fused = torch.cat([g, x_meta], dim=1)
        return self.classifier(fused)

# Training and Evaluation

In [ ]:
# Initialize wandb
WANDB_PROJECT = "First_GNN"
wandb.init(project=WANDB_PROJECT, name='HIBA_Meta_8.2') #WANDB_RUN_NAME
wandb.config.batch_size = BATCH_SIZE
wandb.config.learning_rate = LEARNING_RATE
#wandb.config.weight_decay = WEIGHT_DECAY
wandb.config.epochs = EPOCHS

In [35]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = CustomSAGE_APPNP(node_feat_dim=512, meta_feat_dim=1, hidden_channels=256, out_channels=3).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:
        batch = batch.to(device)
        out = model(batch.x, batch.meta.to(device), batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Calculate Training Accuracy
        preds = out.argmax(dim=1)
        correct += (preds == batch.y).sum().item()
        total += batch.y.size(0)
        
    avg_epoch_loss = total_loss / len(train_loader)
    train_acc = correct / total

    #print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

    # Validation
    model.eval()
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            output = model(batch.x, batch.meta.to(device), batch.edge_index, batch.batch)
            predicted = output.argmax(dim=1) #_, torch.max(output, dim=1)
            correct_val += (predicted == batch.y).sum().item()
            total_val += batch.y.size(0)
            
            #all_preds.extend(predicted.cpu().numpy())
            #all_labels.extend(batch.y.cpu().numpy())
            
    #accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    val_acc = correct_val / total_val
    
    wandb.log({"epoch": epoch + 1, "train_loss": avg_epoch_loss, "train_accuracy": train_acc})
    
    wandb.log({"epoch": epoch + 1, "loss": avg_epoch_loss, "val_accuracy": val_acc})
    
    #print(f"Epoch {epoch+1} - loss: {avg_epoch_loss:.4f} - train_acc: {train_acc:.4f}")
    print(f"Epoch {epoch+1} - loss: {avg_epoch_loss:.4f} - val_acc: {val_acc:.4f}")
    
    
    #if val_acc > best_val_acc:
    #    best_val_acc = val_acc
        #wandb.log({"best_val_accuracy": best_val_acc})

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.meta.to(device), batch.edge_index, batch.batch)
            preds = out.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch.y.cpu().numpy())

            correct += (preds == batch.y).sum().item()
            total += batch.y.size(0)

    # Accuracy
    test_acc = 100 * correct / total
    wandb.log({"Test Accuracy": test_acc})

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    
    # Classification report
    report = classification_report(all_labels, all_preds, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv('classification_report.csv', index=True)

    # W&B classification report table
    columns = ["class", "precision", "recall", "f1-score", "support"]
    table = wandb.Table(columns=columns)

    for class_label, metrics in report.items():
        if isinstance(metrics, dict):
            precision = metrics.get("precision")
            recall = metrics.get("recall")
            f1 = metrics.get("f1-score")
            support = metrics.get("support")
            table.add_data(str(class_label), precision, recall, f1, support)

    wandb.log({"classification_report": table})

    # W&B confusion matrix plot
    class_names = ["Benign", "Malignant"]
    wandb.log({
        "confusion_matrix": wandb.plot.confusion_matrix(
            preds=all_preds,
            y_true=all_labels,
            class_names=class_names
        )
    })

    # Save matplotlib confusion matrix
    plt.figure(figsize=(6, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="flare",
                xticklabels=class_names,
                yticklabels=class_names)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.savefig("confusion_matrix.png")
    plt.close()

    wandb.log({"confusion_matrix_image": wandb.Image("confusion_matrix.png")})
    wandb.finish()

# Example train for one epoch
#train(model, loader, device)
evaluate(model, test_loader, device)